In [11]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
from dotenv import load_dotenv, find_dotenv
from utils.outlier_detection import detect_outliers_by_series
from utils.formatting_functions import *
from utils.db_functions import *
from utils.excel_functions import *

# Setup

In [25]:
# Data path
data_path = Path('C:/Users/alfon/Desktop/UN/Data')

data_files_dict = {'index_1030' : ['index_1030 Gender Development Index.xlsx', '2025_metadata_Gender Development Index 2025.xlsx'],
                   'index_9' : ['index_9 EGDI.xlsx', '20251102_metadata_E-Government Development Index_2024.xlsx', 'raw_EGDI_2024.xlsx'],
                   'index_1032': ['20260710_historical data_index_1032_SDG 2026.xlsx', '20260710_metadata_Sustainable_Development_Index_2026.xlsx', 'raw_SDG_2026.xlsx'],

                    # Somalia indices
                   'index_12' : ['index_12 E-participation Index.xlsx', '', 'raw_EPI_2024.xlsx'],
                   'index_1049' : ['index_1049.xlsx', '', 'raw_IIAG_2024.xlsx'],
                   'index_1037' : ['index_1037.xlsx', '', 'raw_GOVTECH_2022.xlsx'],
                   'index_1022': ['index_1022 Women Peace and security Index.xlsx','','raw_WPS_2025.xlsx'],
                   
                   # Somalia second batch
                   'index_1050': ['index_1050 Global Hunger.xlsx','','raw_GHI.xlsx'],
                   'index_5': ['index_5 ICT Developement Index.xlsx','','raw_ITU_2024.xlsx'],
                   'index_1036': ['20262205_historical data_index_1036_WPF_2026.xlsx','','raw_WPF_2026.xlsx'],

                   # Third
                   'index_1027': ['index_1027 Human Development Index.xlsx','','raw_HDI.xlsx'],
                   'index_1030': ['index_1030 Gender Development Index.xlsx','','raw_GDI_2024.xlsx'],
                   'index_1031': ['index_1031 Planetary adjusted HDI.xlsx','','raw_PPHDI.xlsx'],
                   'index_1035': ['20260303_Historical data _WBL2.0_2026.xlsx','','raw_WBL2.0_2026.xlsx'],
                   # None,
                   'index_1026': ['index_1026 Open data Policies.xlsx','','raw_ODP_2022.xlsx'],
                   'index_1053': ['20260721_historical data_index_1053_BTI2026.xlsx','','raw_BTI_2026.xlsx'],
                   'index_1021': ['index_1021.xlsx','','raw_WBL 1.0.xlsx'],
                   'index_1052': ['20260717_historical data_index_1052_WPJ2026.xlsx','','raw_WJP_2025.xlsx'],
                   'index_1034': ['index_1034.xlsx','','raw_EnviPI_2026.xlsx'],

                   # Check below on Outlier detection
                   'index_1024': ['index_1024_FII.xlsx','','raw_FII_2025 1.xlsx'], # No Somalia

                   'index_1051': ['20263007_historical data_LNOB_index_1051_2026 1.xlsx','','raw_LNOB_2026 1.xlsx'],

                   'index_3': ['index_3.xlsx','','raw_GCYB_2024.xlsx'],
                   'index_7': ['index_7 GII.xlsx','','raw_GII_2025.xlsx'],   # No Somalia
                   'index_10': ['index_10 DARE.xlsx', '', 'raw_DARE 1.xlsx']
                   
                  }

index_name = 'index_1022'
index_data_files = data_files_dict[index_name]

# Read straight from DB
read_from_db = False
db_env = 'prod'
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("No .env file found in the current directory or its parents")
load_dotenv(env_path, override=True)

# Run on all/escwa countries
only_escwa_countries = False 

index_historical_kpi_data_path = Path(data_path) / 'Historical' / index_data_files[0]
index_simulator_data_path = Path(data_path) / 'Simulator Data' / index_data_files[2]
diagnosis_assessment_template_path = Path(data_path) / 'Reference templates' / 'Data_Diagnosis_Assessment_template.xlsx'

# Reference template (for Country Profile)
diagnosis_assessment_template = pd.read_excel(diagnosis_assessment_template_path)

# Read raw data
historical_kpi_data_raw = pd.read_excel(index_historical_kpi_data_path)
simulator_data_raw = pd.read_excel(index_simulator_data_path)

country_iso = pd.read_excel(Path(data_path) / 'metadata_SDG_2026.xlsx', sheet_name='country')
country_iso['country'] = country_iso['country'].str.strip()
country_iso['ISO'] = country_iso['ISO'].str.strip()

# Reference Data
escwa_members = ['DZA','BHR','COM','DJI','EGY','IRQ','JOR','KWT','LBN','LBY','MRT',
                 'MAR','OMN','PSE','QAT','SAU','SOM','SDN','SYR','TUN','ARE','YEM']

kpi_key = (simulator_data_raw.iloc[0]
           .rename('Series Name')
           .rename_axis('KPI ID')
           .reset_index()
           .dropna(subset=['Series Name']))
kpi_key['Series Name'] = kpi_key['Series Name'].astype(str).str.strip()
kpi_key['KPI ID'] = kpi_key['KPI ID'].astype(str).str.strip()
kpi_list = kpi_key['KPI ID'].unique().tolist()

#### Format Simulator and Historical Data

In [26]:
# Read from DB or excel file
if read_from_db == True:
    connection_string = get_connection_string(env=db_env)
    # Simulator Data
    simulator_data = get_latest_sim_data(connection_string, kpi_list)
    
    # Historical KPI Data
    historical_kpi_data_long = get_latest_hist_data(connection_string, kpi_list)
    historical_kpi_data_long['data_historical_automated'] = pd.to_numeric(
        historical_kpi_data_long['data_historical_automated'], errors='coerce').astype('Float64')
    historical_kpi_data_long['KPI ID'] = historical_kpi_data_long['KPI ID'].astype(str).str.strip()
    historical_kpi_data_long['year_historical_automated'] = pd.to_numeric(
        historical_kpi_data_long['year_historical_automated'], errors='coerce').astype('Int64')
    historical_kpi_data_long_escwa = historical_kpi_data_long[historical_kpi_data_long['ISO'].isin(escwa_members)]
    series_to_kpi = historical_kpi_data_long[['Series Name', 'KPI ID']].copy().drop_duplicates().reset_index(drop=True)
else:
    simulator_data = format_simulator_data(simulator_data_raw, kpi_key,
                                           pivot_names=('Country', 'ISO', 'Region', 'Sub-region', 'Income Type'))
    historical_kpi_data_long = format_historical_kpi_data(historical_kpi_data_raw, iso_filter=None)
    historical_kpi_data_long_escwa = historical_kpi_data_long[historical_kpi_data_long['ISO'].isin(escwa_members)]
    
    series_to_kpi = historical_kpi_data_long[['Series Name', 'KPI ID']].copy().drop_duplicates().reset_index(drop=True)

In [27]:
#####################################################
########## Data Checks and Validation ###############
#####################################################

simulator_kpi_ids = simulator_data['KPI ID'].unique().tolist()
historical_data_kpi_ids = historical_kpi_data_long['KPI ID'].unique().tolist()
all_contained = set(historical_data_kpi_ids).issubset(set(simulator_kpi_ids))
missing = set(simulator_kpi_ids) - set(historical_data_kpi_ids)
print(missing)  # empty set if all contained
print(len(missing))


historical_kpi_list = historical_kpi_data_long['KPI ID'].unique().tolist()
simulator_list = simulator_data['KPI ID'].unique().tolist()

hist_set = set(historical_kpi_list)
sim_set = set(simulator_list)

only_in_historical = hist_set - sim_set
only_in_simulator = sim_set - hist_set

print('Only in historical:', only_in_historical)
print('Only in simulator:', only_in_simulator)
print('Same set of values?', hist_set == sim_set)
print('Same length (checks dupes too)?', len(historical_kpi_list) == len(simulator_list))

set()
0
Only in historical: set()
Only in simulator: set()
Same set of values? True
Same length (checks dupes too)? True


#### Reference Tables + Outlier Detection


In [28]:

####################################
##### Simulator Data Cap/Floor #####
####################################
simulator_data_cap = get_simulator_data_cap(simulator_data, historical_kpi_data_long)

####################################
##### Summary KPI/Country data #####
####################################
kpi_country_summary_df = get_kpi_country_summary(historical_kpi_data_long)

#########################################################
##### Match Data - to be used for bucketing records #####
#########################################################
match_data = get_match_data(simulator_data, historical_kpi_data_long, kpi_country_summary_df)

##########################################
#### Newest KPI Data for each country ####
##########################################
newest_kpi_data = historical_kpi_data_long.copy()
newest_kpi_data = newest_kpi_data.dropna(subset=['data_historical_automated'])                            # drop na
newest_kpi_data = newest_kpi_data[
    newest_kpi_data['year_historical_automated'] ==
    newest_kpi_data.groupby(['ISO', 'KPI ID'])['year_historical_automated'].transform('max')]
newest_kpi_data = newest_kpi_data[['KPI ID', 'ISO', 'year_historical_automated', 'data_historical_automated']]
newest_kpi_data = newest_kpi_data.rename(columns={'year_historical_automated':'most_recent_year', 
                                              'data_historical_automated':'most_recent_data'})

#########################################################
################ Detected Outliers ######################
#########################################################

import time
# Start the timer
start_time = time.perf_counter()

# Exclude categorical KPIs
kpi_categorical = simulator_data_cap[simulator_data_cap['unique_categorical_values_simulator'].notna()]['KPI ID'].unique().tolist()
#if only_escwa_countries:
outliers = detect_outliers_by_series(historical_kpi_data_long_escwa[~historical_kpi_data_long_escwa['KPI ID'].isin(kpi_categorical)])
#else:
#    outliers = detect_outliers_by_series(historical_kpi_data_long)

outliers_marker = outliers[outliers['outlier_total']>=3].copy().reset_index(drop=True)
outliers_marker['outlier_detected'] = 1
outliers_marker = outliers_marker[['KPI ID', 'ISO', 'year_historical_automated', 'outlier_detected']]
outliers_marker = outliers_marker.rename(columns={'year_historical_automated':'year_inferred'})

# End the timer
end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Outlier detection execution time: {execution_time:.6f} seconds")

C:\Users\alfon\repos\un\Data Quality\utils\outlier_detection.py:54: RuntimeWarning: overflow encountered in exp
  return b + L / (1 + np.exp(-k * (t - t0)))
C:\Users\alfon\repos\un\Data Quality\utils\outlier_detection.py:54: RuntimeWarning: overflow encountered in exp
  return b + L / (1 + np.exp(-k * (t - t0)))
C:\Users\alfon\repos\un\Data Quality\utils\outlier_detection.py:54: RuntimeWarning: overflow encountered in exp
  return b + L / (1 + np.exp(-k * (t - t0)))
C:\Users\alfon\repos\un\Data Quality\utils\outlier_detection.py:54: RuntimeWarning: overflow encountered in exp
  return b + L / (1 + np.exp(-k * (t - t0)))


Outlier detection execution time: 51.574891 seconds


### Recreate metadata

In [29]:

# Start from simulator data
metadata_automated = simulator_data.copy()
print(f"Baseline Simulator data structure: {metadata_automated.shape}")
baseline_sim_len = metadata_automated.shape[0]

#metadata_automated = metadata_automated.merge(series_to_kpi, how='left',on='Series Name') # Include KPI IDs
metadata_automated = metadata_automated.merge(kpi_country_summary_df, how='left', on=['KPI ID', 'ISO'])
metadata_automated = metadata_automated.merge(simulator_data_cap, how='left', on=['KPI ID'])
metadata_automated = metadata_automated.merge(newest_kpi_data, how='left',on=['KPI ID','ISO']) # Include most recent year
print(f"Structure after additional data: {metadata_automated.shape}")
metadata_sim_len = metadata_automated.shape[0]

if metadata_sim_len != baseline_sim_len:
    print(f"⚠️ Row count changed after merges: {baseline_sim_len} → {metadata_sim_len} "
          f"({metadata_sim_len - baseline_sim_len:+} rows). Check for duplicate keys in the merged tables.")
else:
    print(f"✅ Confirmed: row count unchanged after merges ({metadata_sim_len} rows).")

error_tol = 0.01 # 1%
exact_match = match_data[match_data['exact_match']].copy()
approx_match = match_data[~match_data['exact_match'] &
                          (match_data['value_difference_percentage'] <= error_tol)].copy()

# Col Names
A_exact_year = 'A_exact_year'
A_exact_data = 'A_exact_data'
A_exact_value_diff = 'A_exact_value_diff'
A_exact_value_diff_pct = 'A_exact_value_diff_pct'

A_approx_year = 'A_approx_year'
A_approx_data = 'A_approx_data'
A_approx_value_diff = 'A_approx_value_diff'
A_approx_value_diff_pct = 'A_approx_value_diff_pct'

B_exact_year = 'B_exact_year'
B_exact_data = 'B_exact_data'
B_exact_value_diff = 'B_exact_value_diff'
B_exact_value_diff_pct = 'B_exact_value_diff_pct'

B_approx_year = 'B_approx_year'
B_approx_data = 'B_approx_data'
B_approx_value_diff = 'B_approx_value_diff'
B_approx_value_diff_pct = 'B_approx_value_diff_pct'

A1_cap_adj_year = 'A1_cap_adj_year'
A1_cap_adj_data = 'A1_cap_adj_data'
A1_cap_adj_value_diff = 'A1_cap_adj_value_diff'
A1_cap_adj_value_diff_pct = 'A1_cap_adj_value_diff_pct'

match_cols = ['ISO', 'KPI ID', 'year_historical_automated', 'data_historical_automated','value_difference','value_difference_percentage']

# A - Most Recent Year & Match
# A.1 (Exact Match)
A_exact_match = exact_match[exact_match['latest_year']].copy()
A_exact_match = A_exact_match[match_cols]
A_exact_match = A_exact_match.rename(columns={'year_historical_automated':A_exact_year, 
                                              'data_historical_automated':A_exact_data,
                                              'value_difference':A_exact_value_diff,
                                              'value_difference_percentage':A_exact_value_diff_pct})
# A.2 (Approximate Match)
A_approx_match = approx_match[approx_match['latest_year']].copy()
A_approx_match = A_approx_match[match_cols]
A_approx_match = A_approx_match.rename(columns={'year_historical_automated':A_approx_year, 
                                                'data_historical_automated':A_approx_data,
                                                'value_difference':A_approx_value_diff,
                                                'value_difference_percentage':A_approx_value_diff_pct})
#A_approx_match = A_exact_match[['ISO', 'KPI ID', 'year_historical_automated', 'data_historical_automated']]
#A_exact_match = A_exact_match.rename(columns={'year_historical_automated':'A_1_year', 'data_historical_automated':'A_1_data'})

# B - Older Year & Match
# B.1 (Exact Match)
# - exclude values from A
exclude_keys = pd.concat([A_exact_match[['ISO', 'KPI ID']], A_approx_match[['ISO', 'KPI ID']]]).drop_duplicates()
exclude_index = pd.MultiIndex.from_frame(exclude_keys)
B_exact_match = exact_match[~exact_match['latest_year']].copy()
B_exact_match = B_exact_match[~pd.MultiIndex.from_frame(B_exact_match[['ISO', 'KPI ID']]).isin(exclude_index)]
# - Max year value available
B_exact_match = B_exact_match[
    B_exact_match['year_historical_automated'] ==
    B_exact_match.groupby(['ISO', 'KPI ID'])['year_historical_automated'].transform('max')]
B_exact_match = B_exact_match[match_cols]
B_exact_match = B_exact_match.rename(columns={'year_historical_automated':B_exact_year, 
                                              'data_historical_automated':B_exact_data,
                                              'value_difference':B_exact_value_diff,
                                              'value_difference_percentage':B_exact_value_diff_pct})
# B.2 (Approximate Match)
exclude_keys = pd.concat([A_exact_match[['ISO', 'KPI ID']], A_approx_match[['ISO', 'KPI ID']], B_exact_match[['ISO', 'KPI ID']] ]).drop_duplicates()
exclude_index = pd.MultiIndex.from_frame(exclude_keys)
B_approx_match = approx_match[~approx_match['latest_year']].copy()
B_approx_match = B_approx_match[B_approx_match['value_difference_percentage'] <= error_tol]
B_approx_match = B_approx_match[~pd.MultiIndex.from_frame(B_approx_match[['ISO', 'KPI ID']]).isin(exclude_index)]
# - Max year value available
B_approx_match = B_approx_match[
    B_approx_match['year_historical_automated'] ==
    B_approx_match.groupby(['ISO', 'KPI ID'])['year_historical_automated'].transform('max')]
B_approx_match = B_approx_match[match_cols]
B_approx_match = B_approx_match.rename(columns={'year_historical_automated':B_approx_year, 
                                                'data_historical_automated':B_approx_data,
                                                'value_difference':B_approx_value_diff,
                                                'value_difference_percentage':B_approx_value_diff_pct})

metadata_automated = metadata_automated.merge(A_exact_match, how='left', on=['KPI ID', 'ISO'])
metadata_automated = metadata_automated.merge(A_approx_match, how='left', on=['KPI ID', 'ISO'])
metadata_automated = metadata_automated.merge(B_exact_match, how='left', on=['KPI ID', 'ISO'])
metadata_automated = metadata_automated.merge(B_approx_match, how='left', on=['KPI ID', 'ISO'])



# Matched fields
metadata_automated['A_exact']= np.where(metadata_automated[A_exact_data].isna(),0,1)
metadata_automated['A_approx']= np.where(metadata_automated[A_approx_data].isna(),0,1)
metadata_automated['A_total']= metadata_automated['A_exact']+metadata_automated['A_approx']

metadata_automated['B_exact']= np.where(metadata_automated[B_exact_data].isna(),0,1)
metadata_automated['B_approx']= np.where(metadata_automated[B_approx_data].isna(),0,1)
metadata_automated['B_total']= metadata_automated['B_exact']+metadata_automated['B_approx']

metadata_automated['A_B_match_total'] = metadata_automated['A_total'] + metadata_automated['B_total']


# A_1_cap_adjusted match
cap_adjusted_match = metadata_automated.copy()
mask = (
    (cap_adjusted_match['A_B_match_total'] == 0) &                                 # - No A or B match
    (cap_adjusted_match['cap_present'] == 1) &                                     # - cap_present == 1
    (cap_adjusted_match['most_recent_data'].notna()) &                             # - most_recent_data available
    (cap_adjusted_match['most_recent_data'] > cap_adjusted_match['possible_cap']) &  # - most_recent_data > likely_cap
    (cap_adjusted_match['data_report'] == cap_adjusted_match['possible_cap'])        # - data_report == likely_cap
)
cap_adjusted_match = cap_adjusted_match[mask]
cap_adjusted_match[A1_cap_adj_year] = cap_adjusted_match['most_recent_year']
cap_adjusted_match[A1_cap_adj_data] = cap_adjusted_match['data_report']
cap_adjusted_match[A1_cap_adj_value_diff] = 0     # - Assume values is the same as data_report, 0 diff
cap_adjusted_match[A1_cap_adj_value_diff_pct] = 0
cap_adjusted_match['A1_cap_adjusted_match'] = 1
# - join into df
cap_adjusted_match = cap_adjusted_match[['KPI ID', 'ISO', A1_cap_adj_year, A1_cap_adj_data, 
                                         A1_cap_adj_value_diff, A1_cap_adj_value_diff_pct, 'A1_cap_adjusted_match']]
metadata_automated = metadata_automated.merge(cap_adjusted_match, how='left', on=['KPI ID','ISO'])
metadata_automated['A1_cap_adjusted_match'] = metadata_automated['A1_cap_adjusted_match'].fillna(0)
metadata_automated['A_A1_B_match_total'] = metadata_automated['A_total'] + metadata_automated['B_total'] + metadata_automated['A1_cap_adjusted_match']


#### NO MATCH BUCKETS ####
# Bucket D - possible impute
# No matches (A, A1, or B) & less than or equal to 5 years of historical data
metadata_automated['D_possible_impute'] = (
    (metadata_automated['A_A1_B_match_total'] == 0) &  # No A/B/C matches
    (metadata_automated['most_recent_data'].notna()) & # Historical data exists
    (metadata_automated['nonnull_rows'] <= 5) &        # Less than or equal to 5 yrs historical data
    (metadata_automated['nonnull_pct'] < 50)           # Non-null pct is greater than .50
).fillna(False).astype(int)

no_hist = metadata_automated['most_recent_data'].isna()
no_report = metadata_automated['data_report'].isna()
# Bucket E - No historical data exists
metadata_automated['E_no_historical_data'] = (no_hist & ~no_report).astype(int)
# Bucket F - No report/simulator data
metadata_automated['F_no_report_data'] = (no_report & ~no_hist).astype(int)
# Bucket G - No historical AND report/simulator data
metadata_automated['G_no_historical_and_report_data'] = (no_hist & no_report).astype(int)
# - if there is no report data then clear D to 0
metadata_automated['D_possible_impute'] = np.where(no_report, 0, metadata_automated['D_possible_impute'])



# Bucket C - everything else
flag_cols_e = ['A_A1_B_match_total',
               'D_possible_impute',
               'E_no_historical_data',
               'F_no_report_data',
               'G_no_historical_and_report_data']
metadata_automated['C_data_available_no_match'] = (
    (metadata_automated[flag_cols_e] == 0).all(axis=1)
).fillna(False).astype(int)
# C_material_unexplained_discrepancy
# Values matches for A, B, C_cap_adjusted
metadata_automated['year_inferred'] = metadata_automated[[A_exact_year, A_approx_year, B_exact_year, B_approx_year, A1_cap_adj_year]].bfill(axis=1).iloc[:,0]
metadata_automated['data_inferred'] = metadata_automated[[A_exact_data, A_approx_data, B_exact_data, B_approx_data, A1_cap_adj_data]].bfill(axis=1).iloc[:,0]
metadata_automated['year_difference_inferred'] = metadata_automated['most_recent_year'] - metadata_automated['year_inferred']
metadata_automated['value_difference_inferred'] = metadata_automated[[A_exact_value_diff, A_approx_value_diff, 
                                                                      B_exact_value_diff, B_approx_value_diff, 
                                                                      A1_cap_adj_value_diff]].bfill(axis=1).iloc[:,0]
metadata_automated['value_difference_percentage_inferred'] = metadata_automated[[A_exact_value_diff_pct, A_approx_value_diff_pct, 
                                                                                 B_exact_value_diff_pct, B_approx_value_diff_pct,
                                                                                 A1_cap_adj_value_diff_pct]].bfill(axis=1).iloc[:,0]


# Final Count (A. B, C, D, E, F buckets)
metadata_automated['total_buckets'] = metadata_automated[
    ['A_A1_B_match_total','C_data_available_no_match','D_possible_impute','E_no_historical_data', 'F_no_report_data', 'G_no_historical_and_report_data']
].sum(axis=1)

### Profile Buckets ###
tol_yr_old = 4
# Percent Accurate [A + C]
metadata_automated['Accurate_A_A1'] = metadata_automated['A_total']+metadata_automated['A1_cap_adjusted_match']
# Percent Old (Within 4 years of most recent data, could be within 4 years of most recent KPI data)
metadata_automated['Old_B'] = np.where(
    (metadata_automated['B_total'] == 1) & (metadata_automated['year_difference_inferred'] <= tol_yr_old), 1, 0)
metadata_automated['Very_Old_B'] = np.where(
    (metadata_automated['B_total'] == 1) & (metadata_automated['year_difference_inferred'] > tol_yr_old), 1, 0)
# Percent Unmatched (D, E, F)
metadata_automated['Unmatched_C_D_E_F'] = metadata_automated[
    ['C_data_available_no_match','D_possible_impute','E_no_historical_data', 'F_no_report_data']
].sum(axis=1)
metadata_automated['total_profile'] = metadata_automated[
    ['Accurate_A_A1', 'Old_B', 'Very_Old_B', 'Unmatched_C_D_E_F']
    ].sum(axis=1)

# Merge outliers only on matched data values
metadata_automated = metadata_automated.merge(outliers_marker,how='left')


print(f"Structure after data quality info: {metadata_automated.shape}")
metadata_2_sim_len = metadata_automated.shape[0]

if metadata_2_sim_len != baseline_sim_len:
    print(f"⚠️ Row count changed after data quality merges: {baseline_sim_len} → {metadata_2_sim_len} "
          f"({metadata_2_sim_len - baseline_sim_len:+} rows). Check for duplicate keys in the merged tables.")
else:
    print(f"✅ Confirmed: row count unchanged after data quality merges ({metadata_2_sim_len} rows).")

print("Rows (first datapoint) should match")
#assert (metadata_automated[bucket_cols].sum(axis=1) == 1).all()
violations = metadata_automated[
    (metadata_automated['B_total'] == 1) &
    (metadata_automated['most_recent_year'].isna() | metadata_automated['year_inferred'].isna())
]

if len(violations) > 0:
    print(f"⚠️ {len(violations)} rows have B_total==1 but are missing most_recent_year or year_inferred")
else:
    print("✅ Confirmed: all B_total==1 rows have both most_recent_year and year_inferred.")

Baseline Simulator data structure: (2353, 5)
Structure after additional data: (2353, 20)
✅ Confirmed: row count unchanged after merges (2353 rows).
Structure after data quality info: (2353, 66)
✅ Confirmed: row count unchanged after data quality merges (2353 rows).
Rows (first datapoint) should match
✅ Confirmed: all B_total==1 rows have both most_recent_year and year_inferred.


In [30]:
# For Summary sheet
kpi_year = get_kpi_year_stats(metadata_automated)
kpi_data = get_kpi_totals(metadata_automated).merge(get_kpi_year_stats(metadata_automated), on='KPI ID', how='left')
#kpi_data = kpi_data.merge(simulator_data_cap, how='left', on='KPI ID')


In [31]:
# Final column ordering
base_cols=['Country', 'ISO', 'Series Name','KPI ID','data_report', 'total_rows',
          'nonnull_rows', 'nonnull_pct', 'latest_year_nonnull', 'possible_cap',
          'cap_count','max_kpi_value_hist', 'possible_floor', 'floor_count', 'cap_present',
           'n_unique_categorical_simulator', 'unique_categorical_values_simulator',
           'most_recent_year', 'most_recent_data']
inferred_cols=['year_inferred', 'data_inferred','year_difference_inferred', 'value_difference_inferred','value_difference_percentage_inferred']
totals_cols=['A_exact', 'A_approx','A_total','A1_cap_adjusted_match','B_exact', 'B_approx', 'B_total', 'A_B_match_total','A_A1_B_match_total',
             'C_data_available_no_match', 'D_possible_impute','E_no_historical_data','F_no_report_data','G_no_historical_and_report_data','total_buckets']
profile_totals_cols = ['Accurate_A_A1', 'Old_B', 'Very_Old_B', 'Unmatched_C_D_E_F', 'total_profile']
misc_cols = ['outlier_detected']

# Filter for ESCWA states
if only_escwa_countries:
    metadata_automated_escwa = metadata_automated[metadata_automated['ISO'].isin(escwa_members)].reset_index(drop=True).copy()
else:
    metadata_automated_escwa = metadata_automated.copy()    
metadata_automated_escwa = metadata_automated_escwa[base_cols+inferred_cols+totals_cols+misc_cols]


In [32]:
# Outlier Sheet
detected_outliers = metadata_automated_escwa.copy()
detected_outliers = detected_outliers[detected_outliers['outlier_detected']==1]

# Pull in only series with detected outlier in matched metadata
keys = detected_outliers[['ISO', 'KPI ID', 'year_inferred']]#.drop_duplicates().reset_index(drop=True)
keys['flagged_year'] = 'X'
keys.columns = ['ISO', 'KPI ID', 'year_historical_automated', 'flagged_year']
df_outliers = outliers.merge(keys[['ISO','KPI ID']], on=['ISO', 'KPI ID'], how='inner')
df_outliers = df_outliers.sort_values(['KPI ID', 'ISO', 'year_historical_automated'], 
                                      ascending=[True, True, False]).reset_index(drop=True)
print(df_outliers.shape)
df_outliers_before_len = df_outliers.shape[0]
df_outliers = df_outliers.merge(keys, on=['ISO','KPI ID','year_historical_automated'], how='left')
print(df_outliers.shape)
df_outliers_after_len = df_outliers.shape[0]

if df_outliers_before_len != df_outliers_after_len:
    print(f"⚠️ Row count changed in outlier df: {baseline_sim_len} → {metadata_2_sim_len} "
          f"({df_outliers_before_len - df_outliers_after_len:+} rows). Check for duplicate keys in the merged tables.")
else:
    print(f"✅ Confirmed: row count unchanged after outlier df ({df_outliers_after_len} rows).")


(1834, 23)
(1834, 24)
✅ Confirmed: row count unchanged after outlier df (1834 rows).


# Summary Statisitics

In [33]:
quality_cols = ['A_total',
                'A1_cap_adjusted_match',
                'B_total',
                'C_data_available_no_match',
                'D_possible_impute',
                'E_no_historical_data',
                'F_no_report_data',
                'G_no_historical_and_report_data']

summary_stats, summary_stats_kpi, summary_stats_country = get_summary_stats(
    metadata_automated_escwa, quality_cols, simulator_data,
    only_escwa_countries=only_escwa_countries, escwa_members=escwa_members,
)

✅ All 2353 records accounted for.
Simulator Data (ESCWA members) count 2353


# Metadata Automation Output to file

In [34]:
file_name = f"Metadata_Automation_{index_name}.xlsx"
output_path = Path('C:/Users/alfon/Desktop/UN/Data/Metadata Automation Outputs') / file_name

# only ESCWA countries in long format
if only_escwa_countries:
    data_long_escwa = historical_kpi_data_long_escwa#[historical_kpi_data_long['ISO'].isin(escwa_members)].copy()
else:
    data_long_escwa = historical_kpi_data_long

sheets = {
    'Metadata_automated' : metadata_automated_escwa,
    'KPI Data Summary' : kpi_data,
    'Summary Stats' : summary_stats,
    'Summary Stats by KPI' : summary_stats_kpi,
    'Summary Stats by Country' : summary_stats_country,
    'Raw Data Long format' : data_long_escwa,
    'Outliers data': df_outliers
}
write_dataframes_to_excel(
        sheets,
        output_path=output_path,
        freeze_header=True,
        autofit_columns=True,
        #header_bg_color="4472C4",
        #header_font_color="FFFFFF",
        header_bold=True,
        add_borders=True,
        add_autofilter=True,
        index=False,
        plot_df=df_outliers,          # the frame with the plot columns
    )


Saved: C:\Users\alfon\Desktop\UN\Data\Metadata Automation Outputs\Metadata_Automation_index_1022.xlsx


# Country Profile Output

In [13]:
col_renames = {'A_total'                            : 'Bucket A: Data timely and accurate',
               'A1_cap_adjusted_match'              : 'Bucket A1: Data timely and accurate but capped',
               'B_total'                            : 'Bucket B: Data old but accurate',
               'C_data_available_no_match'          : 'Bucket C: Historical data available but no match',
               'D_possible_impute'                  : 'Bucket D: Expert judgement / imputed',
               'E_no_historical_data'               : 'Bucket E: No Historical data available',
               'F_no_report_data'                   : 'Bucket F: No Report data available',
               'G_no_historical_and_report_data'    : 'Bucket G: No Historical and Report data available'}

quality_cols = ['Bucket A: Data timely and accurate', 
                'Bucket A1: Data timely and accurate but capped', 
                'Bucket B: Data old but accurate', 
                'Bucket C: Historical data available but no match',
                'Bucket D: Expert judgement / imputed',
                'Bucket E: No Historical data available',
                'Bucket F: No Report data available',
                'Bucket G: No Historical and Report data available']

# Static Tables
summary_stats_all = profile_data_quality_summary(metadata_automated.rename(columns=col_renames), quality_cols)
summary_stats_escwa = profile_data_quality_summary(metadata_automated_escwa.rename(columns=col_renames), quality_cols)
summary_stats_kpi = profile_quality_summary_table(metadata_automated.rename(columns=col_renames), quality_cols, groupby=['KPI ID','Series Name'], pct_only=True)

All 2895 records accounted for.
All 2895 records accounted for.


### Write to excel

In [16]:

index_title = index_name
file_name = f"Profile_EGYPT_{index_title}.xlsx"
output_path = Path('C:/Users/alfon/Desktop/UN/Data/Profile Output') / file_name

title = 'General Overview: Overall Quality of Index'
 
# constant across every sheet, in the order they should be stacked
static_tables = {
    'All Countries'                                : summary_stats_all,
    'ESCWA Countries'                              : summary_stats_escwa,
    f'{index_title} - KPI Quality (All Countries)' : summary_stats_kpi,
}

#country_list = metadata_automated_escwa['Country'].unique().tolist()
country_iso_list = ['EGY']
 
write_country_sheets_to_excel(
    metadata_automated_escwa.rename(columns=col_renames),
    kpi_year,
    diagnosis_assessment_template,
    country_iso_list,
    quality_cols,
    static_tables,
    output_path=output_path,
    index_title=index_title,
    title=title,
)

All 15 records accounted for.
Saved: C:\Users\alfon\Desktop\UN\Data\Profile Output\Profile_EGYPT_index_9.xlsx (1 sheet)
